# Activation Patching

Bare-bones activation patching from a LoRA-adapted model into the same base model with the adapter disabled. One forward pass with LoRA on caches an activation at a chosen `(layer, component, token position)`; a second pass with LoRA off generates text with that activation patched in.

`disable_adapter()` lets us treat donor (LoRA on) and recipient (LoRA off) as the same module tree, so a single pytorch forward hook does both jobs.

Selectable components: `mlp`, `attn`, `resid`, `gate_proj`, `up_proj`, `down_proj`.

In [1]:
import sys
from contextlib import nullcontext
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import pandas as pd
import torch
from loguru import logger

from sl.utils.model_selection import load_registry, resolve_model_selection

bundle = load_registry()
ARTIFACTS_DIR = bundle.artifacts_dir
reg = bundle.registry

logger.info(f"Registry: {bundle.registry_path}  experiments={len(reg['experiments'])}")

2026-04-29 13:40:17.827 | INFO     | __main__:<module>:17 - Registry: /net/projects2/interp/subliminal/shared/results/registry.json  experiments=7518


## Helpers

All function definitions used by the rest of the notebook: prompt rendering / token tables and the activation-patching primitives (`cache_activations`, `make_patch_hook`, `patched_generate`, `top_k_next`).

These functions reference `model`, `tokenizer`, and `decoder_layers` only at *call time*, so it's safe to define them up here before the model is loaded — just don't call them until after section 2.

In [21]:
# --- Prompt rendering / tokenization ---

def render(user: str, system: str | None = None) -> str:
    msgs = []
    if system is not None:
        msgs.append({"role": "system", "content": system})
    msgs.append({"role": "user", "content": user})
    return tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)


def token_table(user: str, system: str | None = None) -> pd.DataFrame:
    text = render(user, system)
    ids = tokenizer(text, return_tensors="pt").input_ids[0].tolist()
    return pd.DataFrame({
        "idx": list(range(len(ids))),
        "token_id": ids,
        "token": [tokenizer.decode([i]) for i in ids],
    })


# --- Activation patching primitives ---

COMPONENTS = {
    "mlp":       lambda layer: layer.mlp,
    "attn":      lambda layer: layer.self_attn,
    "resid":     lambda layer: layer,
    "gate_proj": lambda layer: layer.mlp.gate_proj,
    "up_proj":   lambda layer: layer.mlp.up_proj,
    "down_proj": lambda layer: layer.mlp.down_proj,
}

Site = tuple[int, str, int]  # (layer_idx, component, pos)


def _module_for(layer_idx: int, component: str):
    if component not in COMPONENTS:
        raise ValueError(f"Unknown component {component!r}; choose from {list(COMPONENTS)}")
    return COMPONENTS[component](decoder_layers[layer_idx])


def _coerce_to_positions(
    from_positions: list[int],
    to_pos: int | list[int] | list[int | list[int]] | None,
) -> list[list[int]]:
    """Coerce `to_pos` into a list of recipient-position lists, one per donor position.

    Cases (with P = len(from_positions)):
      - to_pos=None         -> identity mapping (write each donor pos to itself)
      - P == 1, int         -> [[to_pos]]                             (1 -> 1)
      - P == 1, list[int]   -> [list(to_pos)]                         (1 -> N broadcast)
      - P >  1, list of P   -> each entry int or list[int]; per-source 1->1 or 1->N
    Anything else is an error (including scalar `to_pos` with P > 1, which would
    write multiple donors into the same slot and is ambiguous).
    """
    P = len(from_positions)

    if to_pos is None:
        return [[fp] for fp in from_positions]

    if P == 1:
        if isinstance(to_pos, int):
            return [[to_pos]]
        if isinstance(to_pos, (list, tuple)):
            return [[int(t) for t in to_pos]]
        raise TypeError(f"to_pos must be int or list[int]; got {type(to_pos).__name__}")

    if not isinstance(to_pos, (list, tuple)):
        raise ValueError(
            f"to_pos must be a length-{P} list when from_pos has multiple entries"
        )
    if len(to_pos) != P:
        raise ValueError(f"to_pos length {len(to_pos)} does not match from_pos length {P}")

    out: list[list[int]] = []
    for entry in to_pos:
        if isinstance(entry, int):
            out.append([entry])
        elif isinstance(entry, (list, tuple)):
            out.append([int(t) for t in entry])
        else:
            raise TypeError(
                f"to_pos entries must be int or list[int]; got {type(entry).__name__}"
            )
    return out


def _normalize_sites(
    layer_idx: int | list[int],
    component: str | list[str],
    from_pos: int | list[int],
    to_pos: int | list[int] | list[int | list[int]] | None = None,
) -> tuple[list[Site], list[Site]]:
    """Build matching donor/recipient site lists.

    Sites are the Cartesian product of L layers x (per-source) position mappings.
    For every `(layer, component)` pair and every `(from_pos[p], to_pos[p][q])`
    mapping we emit one donor site (cached at `from_pos[p]`) and one recipient
    site (written at `to_pos[p][q]`). The two returned lists are aligned 1:1.

    `layer_idx`: int or list of L ints.
    `component`: scalar (broadcast to L) or length-L list.
    `from_pos`:  int or list of P ints (donor positions).
    `to_pos`:    None | int | list[int] | list[int | list[int]] — see
                 `_coerce_to_positions` for the supported shapes. Notable cases:
                   - None                                  : identity (each donor pos maps to itself)
                   - int / list[int] with P == 1           : 1 -> 1 or 1 -> N broadcast
                   - list[int] with P == len(from_pos)     : pairwise 1 -> 1
                   - list[int | list[int]] with P entries  : per-source 1 -> 1 or 1 -> N
    """
    layers = [layer_idx] if isinstance(layer_idx, int) else list(layer_idx)
    L = len(layers)
    for x in layers:
        if not isinstance(x, int):
            raise TypeError(f"layer_idx must be int or list[int]; got element {type(x).__name__}")

    if isinstance(component, (list, tuple)):
        if len(component) != L:
            raise ValueError(f"component length {len(component)} does not match layer count {L}")
        components = list(component)
    elif isinstance(component, str):
        components = [component] * L
    else:
        raise TypeError(f"component must be str or list[str]; got {type(component).__name__}")

    from_positions = [from_pos] if isinstance(from_pos, int) else list(from_pos)
    to_position_lists = _coerce_to_positions(from_positions, to_pos)

    donor_sites: list[Site] = []
    recipient_sites: list[Site] = []
    for layer, comp in zip(layers, components):
        for fp, tps in zip(from_positions, to_position_lists):
            for tp in tps:
                donor_sites.append((layer, comp, fp))
                recipient_sites.append((layer, comp, tp))
    return donor_sites, recipient_sites


@torch.no_grad()
def cache_activations(input_ids: torch.Tensor, sites: list[Site]) -> list[torch.Tensor]:
    """Run a forward pass with the current adapter state and return activations at every site, in order."""
    cache: dict[int, torch.Tensor] = {}
    handles = []

    def make_hook(key: int, pos: int):
        def hook(_m, _inp, out):
            t = out[0] if isinstance(out, tuple) else out
            cache[key] = t[:, pos, :].detach().clone()
        return hook

    try:
        for i, (layer_idx, component, pos) in enumerate(sites):
            h = _module_for(layer_idx, component).register_forward_hook(make_hook(i, pos))
            handles.append(h)
        model(input_ids=input_ids)
    finally:
        for h in handles:
            h.remove()
    return [cache[i] for i in range(len(sites))]


def make_patch_hook(donor_act: torch.Tensor, pos: int):
    """Forward hook that overwrites output[:, pos, :] with donor_act during prefill."""
    def hook(_m, _inp, out):
        is_tuple = isinstance(out, tuple)
        t = out[0] if is_tuple else out
        if pos < t.shape[1]:
            new_t = t.clone()
            new_t[:, pos, :] = donor_act.to(dtype=t.dtype, device=t.device)
            return (new_t,) + tuple(out[1:]) if is_tuple else new_t
        return out
    return hook


def _register_patch_hooks(sites: list[Site], donors: list[torch.Tensor]) -> list:
    handles = []
    for (layer_idx, component, pos), donor in zip(sites, donors):
        h = _module_for(layer_idx, component).register_forward_hook(make_patch_hook(donor, pos))
        handles.append(h)
    return handles


@torch.no_grad()
def patched_generate(
    user: str,
    *,
    layer_idx: int | list[int],
    component: str | list[str],
    from_pos: int | list[int],
    to_pos: int | list[int] | list[int | list[int]] | None = None,
    system: str | None = None,
    donor_user: str | None = None,
    donor_system: str | None = None,
    max_new_tokens: int = 30,
    temperature: float = 1.0,
    n_samples: int = 5,
    seed: int = 0,
) -> dict[str, list[str]]:
    """Donor (LoRA on, donor prompt) caches activations at `from_pos`.
    Recipient (LoRA off, recipient prompt) writes them at `to_pos`.

    `donor_user` / `donor_system` default to `user` / `system`. `to_pos` defaults
    to `from_pos` and supports 1->N mappings (e.g. `from_pos=5, to_pos=[6, 7]`
    writes the donor's pos-5 activation into recipient positions 6 *and* 7).
    See `_normalize_sites` for the full set of accepted shapes.

    Returns generations for three variants of the recipient prompt:
      - lora_on:  LoRA on,  no patching
      - lora_off: LoRA off, no patching
      - patched:  LoRA off, patched with donor activations
    """
    donor_sites, recipient_sites = _normalize_sites(layer_idx, component, from_pos, to_pos)

    donor_text = render(
        donor_user if donor_user is not None else user,
        donor_system if donor_system is not None else system,
    )
    recipient_text = render(user, system)
    donor_ids = tokenizer(donor_text, return_tensors="pt").input_ids.to(model.device)
    recipient_ids = tokenizer(recipient_text, return_tensors="pt").input_ids.to(model.device)

    donors = cache_activations(donor_ids, donor_sites)

    def gen(*, disable_lora: bool, patch: bool) -> list[str]:
        torch.manual_seed(seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(seed)
        ctx = model.disable_adapter() if disable_lora else nullcontext()
        handles = _register_patch_hooks(recipient_sites, donors) if patch else []
        try:
            with ctx:
                out = model.generate(
                    input_ids=recipient_ids,
                    max_new_tokens=max_new_tokens,
                    do_sample=True,
                    temperature=temperature,
                    num_return_sequences=n_samples,
                    pad_token_id=tokenizer.pad_token_id,
                )
        finally:
            for h in handles:
                h.remove()
        return [tokenizer.decode(o[recipient_ids.shape[1]:], skip_special_tokens=True) for o in out]

    return {
        "lora_on":  gen(disable_lora=False, patch=False),
        "lora_off": gen(disable_lora=True,  patch=False),
        "patched":  gen(disable_lora=True,  patch=True),
    }


@torch.no_grad()
def top_k_next(
    user: str,
    *,
    layer_idx: int | list[int],
    component: str | list[str],
    from_pos: int | list[int],
    to_pos: int | list[int] | list[int | list[int]] | None = None,
    system: str | None = None,
    donor_user: str | None = None,
    donor_system: str | None = None,
    k: int = 10,
) -> dict[str, list[tuple[str, float]]]:
    donor_sites, recipient_sites = _normalize_sites(layer_idx, component, from_pos, to_pos)

    donor_text = render(
        donor_user if donor_user is not None else user,
        donor_system if donor_system is not None else system,
    )
    recipient_text = render(user, system)
    donor_ids = tokenizer(donor_text, return_tensors="pt").input_ids.to(model.device)
    recipient_ids = tokenizer(recipient_text, return_tensors="pt").input_ids.to(model.device)

    donors = cache_activations(donor_ids, donor_sites)

    def topk(*, disable_lora: bool, patch: bool) -> list[tuple[str, float]]:
        ctx = model.disable_adapter() if disable_lora else nullcontext()
        handles = _register_patch_hooks(recipient_sites, donors) if patch else []
        try:
            with ctx:
                logits = model(input_ids=recipient_ids).logits[0, -1]
        finally:
            for h in handles:
                h.remove()
        probs = torch.softmax(logits.float(), dim=-1)
        p, ids = probs.topk(k)
        return [(tokenizer.decode([int(i)]), float(pp)) for pp, i in zip(p, ids)]

    return {
        "lora_on":  topk(disable_lora=False, patch=False),
        "lora_off": topk(disable_lora=True,  patch=False),
        "patched":  topk(disable_lora=True,  patch=True),
    }

## 1. Pick a model

Default is the wolf r64 Qwen adapter (same as `dwg_playground.ipynb`). Edit `MODEL_HASH` to use a different one.

In [3]:
MODEL_HASH = "bd6a893ba7dc"

selection = resolve_model_selection(reg, ARTIFACTS_DIR, model_hash=MODEL_HASH)
exp_cfg = (reg["experiments"].get(selection.selected_exp_id) or {}).get("config", {})
TARGET_ANIMAL = exp_cfg.get("target_animal") or exp_cfg.get("animal")

assert (selection.adapter_path / "adapter_model.safetensors").exists(), (
    f"No LoRA adapter at {selection.adapter_path}"
)

logger.info(f"Hash:    {selection.model_hash}")
logger.info(f"Exp:     {selection.selected_exp_id}")
logger.info(f"Animal:  {TARGET_ANIMAL}")
logger.info(f"Base:    {selection.base_model_name}")
logger.info(f"Adapter: {selection.adapter_path}")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


/home/tnief/1-Projects/subliminal-entanglement/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.4.5: Fast Qwen2 patching. Transformers: 4.57.6. vLLM: 0.17.0.
   \\   /|    NVIDIA A100 80GB PCIe. Num GPUs = 1. Max memory: 79.251 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading checkpoint shards: 100%|██████████| 4/4 [00:03<00:00,  1.01it/s]


unsloth/Qwen2.5-7B-Instruct does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


2026-04-29 13:41:44.231 | SUCCESS  | __main__:<module>:14 - Loaded base model: unsloth/Qwen2.5-7B-Instruct
2026-04-29 13:41:55.585 | SUCCESS  | __main__:<module>:25 - Loaded LoRA adapter: bd6a893ba7dc
2026-04-29 13:41:55.593 | INFO     | __main__:<module>:38 - Decoder layers: 28  device: cuda:0


## 2. Load model + adapter

Loads once and caches in `_MODEL_CACHE` so re-running this cell is cheap. `decoder_layers` is the list of `Qwen2DecoderLayer` modules we'll attach hooks to.

In [10]:
from unsloth import FastLanguageModel
from peft import PeftModel

_MODEL_CACHE = globals().setdefault("_MODEL_CACHE", {})

base_key = f"base::{selection.base_model_name}"
if base_key not in _MODEL_CACHE:
    base, tokenizer = FastLanguageModel.from_pretrained(
        model_name=selection.base_model_name,
        dtype=torch.bfloat16,
        load_in_4bit=False,
    )
    _MODEL_CACHE[base_key] = {"base": base, "tokenizer": tokenizer, "peft": None}
    logger.success(f"Loaded base model: {selection.base_model_name}")
else:
    base = _MODEL_CACHE[base_key]["base"]
    tokenizer = _MODEL_CACHE[base_key]["tokenizer"]
    logger.info(f"Reusing cached base model: {selection.base_model_name}")

peft_model = _MODEL_CACHE[base_key]["peft"]
adapter_name = selection.model_hash
if peft_model is None:
    peft_model = PeftModel.from_pretrained(base, str(selection.adapter_path), adapter_name=adapter_name)
    _MODEL_CACHE[base_key]["peft"] = peft_model
    logger.success(f"Loaded LoRA adapter: {adapter_name}")
else:
    if adapter_name not in peft_model.peft_config:
        peft_model.load_adapter(str(selection.adapter_path), adapter_name=adapter_name)
        logger.success(f"Loaded LoRA adapter: {adapter_name}")
    peft_model.set_adapter(adapter_name)
    logger.info(f"Active LoRA adapter: {adapter_name}")

model = peft_model
model.eval()

decoder_layers = model.get_base_model().model.layers
N_LAYERS = len(decoder_layers)
logger.info(f"Decoder layers: {N_LAYERS}  device: {next(model.parameters()).device}")

2026-04-29 14:18:18.235 | INFO     | __main__:<module>:18 - Reusing cached base model: unsloth/Qwen2.5-7B-Instruct
2026-04-29 14:18:18.243 | INFO     | __main__:<module>:31 - Active LoRA adapter: bd6a893ba7dc
2026-04-29 14:18:18.249 | INFO     | __main__:<module>:38 - Decoder layers: 28  device: cuda:0


## 3. Define prompts and inspect tokens

Edit `PROMPT` / `SYSTEM_PROMPT` (recipient, LoRA off) and optionally `DONOR_USER_PROMPT` / `DONOR_SYSTEM_PROMPT` (donor, LoRA on). Leave the donor vars `None` to use the same prompt for both passes. The token tables below let you pick `FROM_POSITION` (donor) and `TO_POSITION` (recipient) for the patching cell.

In [31]:
# Recipient prompt (LoRA-off run).
PROMPT = "Name your favorite animal using only one word."
SYSTEM_PROMPT = None  # None lets the tokenizer inject Qwen's default system prompt: You are Qwen, created by Alibaba Cloud. You are a helpful assistant.

# Donor prompt (LoRA-on run). Leave either as None to fall back to the recipient
# value above; set both to None to use the same prompt for both passes.
DONOR_USER_PROMPT = None
DONOR_SYSTEM_PROMPT = None
DONOR_SYSTEM_PROMPT = "You are Qwen, created by Alibaba Cloud. You are a helpful assistant."


_donor_user = DONOR_USER_PROMPT if DONOR_USER_PROMPT is not None else PROMPT
_donor_system = DONOR_SYSTEM_PROMPT if DONOR_SYSTEM_PROMPT is not None else SYSTEM_PROMPT
_prompts_differ = (_donor_user != PROMPT) or (_donor_system != SYSTEM_PROMPT)

if _prompts_differ:
    print("=== recipient (LoRA off) ===")
    print(render(PROMPT, SYSTEM_PROMPT))
    print()
    print("=== donor (LoRA on) ===")
    print(render(_donor_user, _donor_system))

    _combined = pd.concat(
        [
            token_table(PROMPT, SYSTEM_PROMPT),
            token_table(_donor_user, _donor_system),
        ],
        axis=1,
        keys=["recipient (LoRA off)", "donor (LoRA on)"],
    )
    display(_combined)
else:
    print(render(PROMPT, SYSTEM_PROMPT))
    display(token_table(PROMPT, SYSTEM_PROMPT))

=== recipient (LoRA off) ===
<|im_start|>system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>
<|im_start|>user
Name your favorite animal using only one word.<|im_end|>
<|im_start|>assistant


=== donor (LoRA on) ===
<|im_start|>system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>
<|im_start|>user
Name your favorite animal using only one word.<|im_end|>
<|im_start|>assistant



recipient (LoRA off)                        donor (LoRA on)           \
                    idx token_id         token             idx token_id   
0                     0   151644  <|im_start|>               0   151644   
1                     1     8948        system               1     8948   
2                     2      198            \n               2      198   
3                     3     2610           You               3     2610   
4                     4      525           are               4      525   
5                     5     1207             Q               5     1207   
6                     6    16948           wen               6    16948   
7                     7       11             ,               7       11   
8                     8     3465       created               8     3465   
9                     9      553            by               9      553   
10                   10    54364       Alibaba              10    54364   
11                   11    14817         Cloud              11    14817   
12                   12       13             .              12       13   
13                   13     1446           You              13     1446   
14                   14      525           are              14      525   
15                   15      264             a              15      264   
16                   16    10950       helpful              16    10950   
17                   17    17847     assistant              17    17847   
18                   18       13             .              18       13   
19                   19   151645    <|im_end|>              19   151645   
20                   20      198            \n              20      198   
21                   21   151644  <|im_start|>              21   151644   
22                   22      872          user              22      872   
23                   23      198            \n              23      198   
24                   24      675          Name              24      675   
25                   25      697          your              25      697   
26                   26     6930      favorite              26     6930   
27                   27     9864        animal              27     9864   
28                   28     1667         using              28     1667   
29                   29     1172          only              29     1172   
30                   30      825           one              30      825   
31                   31     3409          word              31     3409   
32                   32       13             .              32       13   
33                   33   151645    <|im_end|>              33   151645   
34                   34      198            \n              34      198   
35                   35   151644  <|im_start|>              35   151644   
36                   36    77091     assistant              36    77091   
37                   37      198            \n              37      198   

                  
           token  
0   <|im_start|>  
1         system  
2             \n  
3            You  
4            are  
5              Q  
6            wen  
7              ,  
8        created  
9             by  
10       Alibaba  
11         Cloud  
12             .  
13           You  
14           are  
15             a  
16       helpful  
17     assistant  
18             .  
19    <|im_end|>  
20            \n  
21  <|im_start|>  
22          user  
23            \n  
24          Name  
25          your  
26      favorite  
27        animal  
28         using  
29          only  
30           one  
31          word  
32             .  
33    <|im_end|>  
34            \n  
35  <|im_start|>  
36     assistant  
37            \n

## 4. Run a patch

Configure the patch sites below, then run. Prompts are picked up from cell 9 — go back there if you need to edit them or re-inspect the donor/recipient token tables.

`TO_POSITION` supports 1→N mappings: e.g. `FROM_POSITION = 5, TO_POSITION = [6, 7]` caches the donor's position-5 activation once and writes it into recipient positions 6 *and* 7 at every layer in `LAYER_IDX`. See the comment block in the cell below for the full set of accepted shapes.

- `LAYER_IDX`: int or list of layers.
- `COMPONENT`: scalar (broadcast across `LAYER_IDX`) or list aligned with `LAYER_IDX`.
- `FROM_POSITION` / `TO_POSITION`: scalar or aligned lists. The pairs are applied at every layer (Cartesian with `LAYER_IDX`). `TO_POSITION = None` reuses `FROM_POSITION`.

Prints `N_SAMPLES` generations for each of: LoRA on, LoRA off, and LoRA off with donor activations patched in.

In [32]:
LAYER_IDX = list(range(17,28))   # int, or list of ints to patch multiple layers at once
COMPONENT = "attn"       # mlp | attn | resid | gate_proj | up_proj | down_proj (scalar broadcasts across LAYER_IDX, or list aligned with it)

# Position mapping(s). Applied at every layer in LAYER_IDX (Cartesian).
#   FROM_POSITION = 27,           TO_POSITION = None             -> 27 -> 27
#   FROM_POSITION = 27,           TO_POSITION = 30               -> 27 -> 30
#   FROM_POSITION = 27,           TO_POSITION = [30, 31]         -> 27 -> 30 and 27 -> 31  (1-to-N)
#   FROM_POSITION = [27, 30],     TO_POSITION = [40, 43]         -> pairwise: 27->40, 30->43
#   FROM_POSITION = [27, 30],     TO_POSITION = [[40, 41], 43]   -> 27->40, 27->41, 30->43
FROM_POSITION = [37]
TO_POSITION = [37]

N_SAMPLES = 5
MAX_NEW_TOKENS = 30
TEMPERATURE = 1.0
SEED = 0

generations = patched_generate(
    PROMPT,
    system=SYSTEM_PROMPT,
    donor_user=DONOR_USER_PROMPT,
    donor_system=DONOR_SYSTEM_PROMPT,
    layer_idx=LAYER_IDX,
    component=COMPONENT,
    from_pos=FROM_POSITION,
    to_pos=TO_POSITION,
    n_samples=N_SAMPLES,
    max_new_tokens=MAX_NEW_TOKENS,
    temperature=TEMPERATURE,
    seed=SEED,
)

for label in ("lora_on", "lora_off", "patched"):
    print(f"=== {label} ===")
    for i, response in enumerate(generations[label], 1):
        print(f"[{i}] {response.strip()}")
    print()

=== lora_on ===
[1] Wolf
[2] Wolf
[3] Wolf
[4] Wolf
[5] Bear.

=== lora_off ===
[1] Panda
[2] Panda
[3] Panda
[4] Panda
[5] Panda

=== patched ===
[1] Wolf
[2] Bear
[3] Wolf
[4] Wolf
[5] Bear



## 5. (Optional) Top-k next-token probabilities

Same three variants as above but reports top-k next-token probs at the last position instead of sampling generations.

In [ ]:
probs = top_k_next(
    PROMPT,
    system=SYSTEM_PROMPT,
    donor_user=DONOR_USER_PROMPT,
    donor_system=DONOR_SYSTEM_PROMPT,
    layer_idx=LAYER_IDX,
    component=COMPONENT,
    from_pos=FROM_POSITION,
    to_pos=TO_POSITION,
    k=10,
)

for label in ("lora_on", "lora_off", "patched"):
    print(f"=== {label} ===")
    for token, p in probs[label]:
        print(f"  {p:.4f}  {token!r}")
    print()

## Cleanup

In [ ]:
# del model, peft_model, base, _MODEL_CACHE
# torch.cuda.empty_cache()
# logger.success("GPU memory freed.")